In [2]:
# ---------------------------------------------------------------------------
# Jugador
# ---------------------------------------------------------------------------
class Jugador:
    HP_MAXIMA = 50

    def __init__(self, nombre, nivel=1, clase=None):
        self.nombre = nombre
        self.nivel = nivel
        self.experiencia = 0
        self.oro = 0
        self.hp_maxima = self.HP_MAXIMA
        self.hp = self.hp_maxima
        self.clase = clase if clase is not None else Aprendiz()
        self.clan = None
        self.inventario = []      # Equipos adquiridos en la tienda

    # Experiencia necesaria para pasar del nivel actual al siguiente
    def exp_para_subir(self):
        return 50 * self.nivel

    # Agrega experiencia (ya con modificadores aplicados) y sube de nivel si corresponde
    def ganar_experiencia(self, cantidad):
        if cantidad > 0:
            self.experiencia += cantidad
            while self.experiencia >= self.exp_para_subir():
                self.experiencia -= self.exp_para_subir()
                self._subir_nivel()

    def _subir_nivel(self):
        self.nivel += 1
        self.curar(self.clase.curacion_al_subir_nivel(self.hp_maxima))

    def curar(self, cantidad):
        if cantidad > 0:
            self.hp = min(self.hp_maxima, self.hp + cantidad)

    # Recibe daño base; la clase puede reducirlo. Devuelve el daño realmente recibido
    def recibir_dano(self, base):
        dano = self.clase.modificar_dano(base)
        self.hp = max(0, self.hp - dano)
        return dano

    # Aplica una recompensa {"experiencia", "oro", "vida"} usando los modificadores de la clase
    def aplicar_recompensa(self, recompensa):
        if recompensa is None:
            return False
        self.ganar_experiencia(self.clase.modificar_exp(recompensa["experiencia"]))
        self.oro += self.clase.modificar_oro(recompensa["oro"])
        vida = recompensa["vida"]
        if vida > 0:
            self.curar(vida)
        elif vida < 0:
            self.recibir_dano(-vida)
        return True

    # Completa una tarea (o hábito positivo). Devuelve False si ya estaba completada hoy
    def completar_tarea(self, tarea):
        return self.aplicar_recompensa(tarea.completar())

    # Marca un hábito como "Positivo" o "Negativo"
    def marcar_habito(self, habito, forma="Positivo"):
        if forma == "Positivo":
            recompensa = habito.marcar_positivo()
        elif forma == "Negativo":
            recompensa = habito.marcar_negativo()
        else:
            raise ValueError("La forma debe ser 'Positivo' o 'Negativo'.")
        return self.aplicar_recompensa(recompensa)

    # Muestra los datos del jugador
    def mostrar_datos(self):
        print(f"Jugador: {self.nombre}")
        print(f"Clase: {self.clase.nombre}")
        print(f"Nivel: {self.nivel}")
        print(f"Experiencia: {self.experiencia}/{self.exp_para_subir()}")
        print(f"Vida: {self.hp}/{self.hp_maxima}")
        print(f"Oro: {self.oro}")
        print(f"Clan: {self.clan.nombre if self.clan else 'Sin clan'}")
        print(f"Inventario: {', '.join(i.nombre for i in self.inventario) or 'Vacío'}")


# ---------------------------------------------------------------------------
# Clanes
# ---------------------------------------------------------------------------
# Representa un clan de jugadores (base de ClanAbierto y ClanPrivado)
class Clan:
    def __init__(self, nombre, capacidad=10):
        self.nombre = nombre
        self.capacidad = capacidad
        self.miembros = []

    def tiene_espacio(self):
        return len(self.miembros) < self.capacidad

    # Agrega un jugador si hay espacio, no es miembro y no pertenece a otro clan
    def agregar_miembro(self, jugador):
        if not self.tiene_espacio() or jugador in self.miembros or jugador.clan is not None:
            return False
        self.miembros.append(jugador)
        jugador.clan = self
        return True

    # Muestra los miembros del clan
    def mostrar_miembros(self):
        print(f"Clan: {self.nombre}")
        for jugador in self.miembros:
            print(f"- {jugador.nombre} (Nivel {jugador.nivel}, {jugador.clase.nombre})")


# Representa un clan al que cualquiera puede solicitar ingreso
class ClanAbierto(Clan):
    # Permite ingresar mientras haya espacio
    def unirse(self, jugador):
        return self.agregar_miembro(jugador)


# Representa un clan que requiere invitación
class ClanPrivado(Clan):
    def __init__(self, nombre, capacidad=10):
        super().__init__(nombre, capacidad)
        self.invitados = []

    # Registra una invitación para un jugador
    def invitar(self, jugador):
        if jugador not in self.miembros and jugador not in self.invitados:
            self.invitados.append(jugador)

    # En un clan privado solo se ingresa con invitación
    def agregar_miembro(self, jugador):
        return self.aceptar_invitacion(jugador)

    # Permite ingresar a un jugador invitado
    def aceptar_invitacion(self, jugador):
        if jugador not in self.invitados:
            return False
        if super().agregar_miembro(jugador):
            self.invitados.remove(jugador)
            return True
        return False


In [3]:
# Definimos _redondear para evitar errores al redondear con round()
# round(2.5) = 2 -> No es conveniente para nuestro escenario
def _redondear(valor):
    return int(valor + 0.5)

class ClasePersonaje:
    def __init__(self, nombre, descripcion, mult_exp=1.0, mult_oro=1.0,
                 reduccion_dano=0.0, curacion_nivel=0.5):
        self.__nombre = nombre
        self.__descripcion = descripcion
        self.__mult_exp = mult_exp                # 1.0 = sin bonificación
        self.__mult_oro = mult_oro                # 1.0 = sin bonificación
        self.__reduccion_dano = reduccion_dano    
        self.__curacion_nivel = curacion_nivel    

    @property
    def nombre(self):
        return self.__nombre

    @property
    def descripcion(self):
        return self.__descripcion

    @property
    def mult_exp(self):
        return self.__mult_exp

    @property
    def mult_oro(self):
        return self.__mult_oro

    @property
    def reduccion_dano(self):
        return self.__reduccion_dano

    @property
    def curacion_nivel(self):
        return self.__curacion_nivel

    def __str__(self):
        return f"{self.__nombre}: {self.__descripcion}"

    def modificar_exp(self, base):
        return _redondear(base * self.__mult_exp)

    def modificar_oro(self, base):
        return _redondear(base * self.__mult_oro)

    def modificar_dano(self, base):
        # Devuelve el daño final que recibe el jugador
        if base <= 0:                # sin daño (o valor inválido)
            return 0
        # (1 - reduccion_dano) es la fracción de daño que recibe el jugador
        # max(1, ...) asegura que, si hubo daño, se reciba al menos 1
        return max(1, _redondear(base * (1 - self.__reduccion_dano)))

    def curacion_al_subir_nivel(self, hp_maxima):
        return _redondear(hp_maxima * self.__curacion_nivel)


# Clases hijas
class Aprendiz(ClasePersonaje):
    def __init__(self):
        super().__init__("Aprendiz", "Clase inicial, sin bonificaciones")


class Guerrero(ClasePersonaje):
    def __init__(self):
        super().__init__("Guerrero", "Muy resistente contra el daño", reduccion_dano=0.30)


class Mago(ClasePersonaje):
    def __init__(self):
        super().__init__("Mago", "Sube de nivel más rápido", mult_exp=1.25)


class Sanador(ClasePersonaje):
    def __init__(self):
        super().__init__("Sanador", "Restaura su salud al subir de nivel", curacion_nivel=1.0)


class Picaro(ClasePersonaje):
    def __init__(self):
        super().__init__("Picaro", "Obtiene oro más rápido", mult_oro=1.25)

In [4]:
class ItemTienda:
    # Atributo de clase autoincremental: cada ítem recibe un código único
    _contador_codigo = 1

    def __init__(self, nombre, precio, nivel_requerido, descripcion):
        # Se valida antes de asignar el código para no "gastar" códigos en ítems inválidos
        if precio < 0:
            raise ValueError("El precio no puede ser negativo.")
        if nivel_requerido < 1:
            raise ValueError("El nivel requerido debe ser al menos 1.")
        self.codigo = ItemTienda._contador_codigo
        ItemTienda._contador_codigo += 1
        self.nombre = nombre
        self.precio = precio
        self.nivel_requerido = nivel_requerido
        self.descripcion = descripcion

    # Aplica el efecto del ítem al jugador. Devuelve True si se aplicó (y por tanto se cobra)
    def aplicar(self, jugador):
        raise NotImplementedError("Cada tipo de ítem define su propio efecto.")


class Equipo(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, bonificacion):
        if bonificacion <= 0:
            raise ValueError("La bonificación debe ser positiva.")
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.bonificacion = bonificacion    # puntos que aumentan la hp_maxima del jugador

    def aplicar(self, jugador):
        # Se guarda en el inventario y mejora la vida máxima (sin repetir el mismo equipo)
        if self in jugador.inventario:
            print(f"Ya tienes equipado: {self.nombre}.")
            return False
        jugador.inventario.append(self)
        jugador.hp_maxima += self.bonificacion
        print(f"Has equipado: {self.nombre}. Bonificación: +{self.bonificacion} HP máxima")
        return True


class Pocion(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, hp_restaurada):
        if hp_restaurada <= 0:
            raise ValueError("La vida restaurada debe ser positiva.")
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.hp_restaurada = hp_restaurada

    def aplicar(self, jugador):
        # Cura al jugador respetando su hp_maxima (usa Jugador.curar)
        if jugador.hp >= jugador.hp_maxima:
            print("Tu salud ya está al máximo; no se usó la poción.")
            return False
        hp_antes = jugador.hp
        jugador.curar(self.hp_restaurada)
        print(f"Te has curado {jugador.hp - hp_antes} HP. Salud actual: {jugador.hp}/{jugador.hp_maxima}")
        return True


class Recompensa(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, creada_por_usuario):
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.creada_por_usuario = creada_por_usuario

    def aplicar(self, jugador):
        # Solo se registra el premio real
        print(f"Recompensa canjeada: {self.nombre}. ¡Disfruta tu premio!")
        return True


class Tienda:
    def __init__(self):
        self.items = []    # Lista de objetos ItemTienda

    def agregar_item(self, item):
        if not isinstance(item, ItemTienda):
            raise TypeError("Solo se pueden agregar objetos de tipo ItemTienda.")
        if item not in self.items:
            self.items.append(item)

    def mostrar_tabla(self):
        print("\n--- OBJETOS DE LA TIENDA ---")
        if not self.items:
            print("La tienda está vacía.")
            return
        for item in self.items:
            print(f"[{item.codigo}] {item.nombre} | Nivel req: {item.nivel_requerido} | Precio: {item.precio} oro")
            print(f"    Desc: {item.descripcion}")

    def buscar_por_codigo(self, codigo):
        for item in self.items:
            if item.codigo == codigo:
                return item
        return None

    # Compra un ítem: valida existencia, nivel y oro; cobra solo si el ítem se aplicó
    def comprar(self, jugador, codigo):
        item = self.buscar_por_codigo(codigo)
        if item is None:
            print(f"No existe un ítem con código {codigo}.")
            return False
        if jugador.nivel < item.nivel_requerido:
            print(f"Necesitas nivel {item.nivel_requerido} para comprar {item.nombre}.")
            return False
        if jugador.oro < item.precio:
            print(f"Oro insuficiente para {item.nombre}: tienes {jugador.oro} y cuesta {item.precio}.")
            return False
        if not item.aplicar(jugador):
            return False
        jugador.oro -= item.precio
        print(f"Compra realizada: {item.nombre} (-{item.precio} oro). Oro restante: {jugador.oro}")
        return True

In [ ]:
# Clase general Calendario_Racha con Tarea y sus tres hijas: Habito, Diaria, Pendiente
# La clase Tarea y sus hijas crean, modifican y acceden a datos que representan las tareas de un Jugador.

# Importamos paquetes necesarios
from datetime import datetime
import calendar
import re

# Incluimos una variable FORMATO_FECHA auxiliar para el segundo argumento de strptime
# Incluimos una función hoy() para conocer la fecha actual del sistema
FORMATO_FECHA = "%Y-%m-%d"
def hoy():
    return datetime.now().date()

# ---------------------------------------------------------------------------
# Tarea e Hijos
# ---------------------------------------------------------------------------
class Tarea():
    """Representa una tarea completada o sin completar de un Jugador.
    
    Atributos:
        titulo: El nombre de la tarea.
        dificultad: El nivel de dificultad de la tarea ('Facil', 'Intermedio' o 'Dificil').
        descripcion: Detalles adicionales sobre la tarea.
        historial: Lista de fechas en las que la tarea fue completada.
        fecha_creacion: Fecha en la que se creo un objeto de la clase Tarea.
    """
    # Se define una variable diccionario RECOMPENSAS para cada tipo de dificultad.
    RECOMPENSAS = {"Facil": {"experiencia":6, "oro": 1, "vida":0},
                   "Intermedio": {"experiencia":8, "oro":1, "vida":1},
                   "Dificil": {"experiencia":12, "oro": 3, "vida":4}}
    
    def __init__(self, titulo:str, dificultad:str, descripcion:str = None):
        # Se valida que la dificultad sea una válida.
        if dificultad in self.RECOMPENSAS.keys():
            self.titulo = titulo
            self.dificultad = dificultad
            self.descripcion = descripcion
            self.historial = []
            self.fecha_creacion = str(hoy())
        else:
            raise ValueError(f"Se añadio una dificultad no válida.\n Debe ser una entre {list(self.RECOMEPNSAS.keys())}")

    def editar_tarea(self, nuevo_titulo = None, nueva_dificultad = None, nueva_descripcion = None):
        # Cada sentencia if verifica si se modifico un atributo 
        if nuevo_titulo is not None:
            self.titulo = nuevo_titulo
        if nueva_dificultad is not None:
            # Verifica que la nueva dificultad sea una válida
            if nueva_dificultad not in self.RECOMPENSAS.keys():
                raise ValueError(f"Se añadio una dificultad no válida.\n Debe ser una entre {list(self.RECOMPENSAS.keys())}")
            self.dificultad = nueva_dificultad
        if nueva_descripcion is not None:
            self.descripcion = nueva_descripcion
    
    def calcular_recompensa(self):
        # Devuelve la recompensa apropiada según la dificultad de la Tarea
        return dict(self.RECOMPENSAS[self.dificultad])

    def completar(self):
        hoy1 = str(hoy())
        # Agrega la fecha de hoy en el historial de completaciones en caso no haya sido agregada anteriormente
        if hoy1 not in self.historial:
            self.historial.append(hoy1)
            return self.calcular_recompensa()
        return None
        
    def __repr__(self):
        # Devuelve información de la tarea al momento de llamar al objeto
        recompensa = self.calcular_recompensa()
        return f" ---------- Información sobre la Tarea ---------- \n| Título: {self.titulo} | Dificultad: {self.dificultad} | Descripcion: {self.descripcion} | Fecha de Creación: {self.fecha_creacion} | Nro. de Fechas Cumplidas: {len(self.historial)} | \n\n--- Información sobre la Recompensa ---\n| Experiencia: {recompensa["experiencia"]} | Oro: {recompensa["oro"]} | Vida: {recompensa["vida"]} |"

class Habito(Tarea):
    '''Representa un Habito positivo o negativo que forma un Jugador

    Atributos:
        titulo: El nombre del habito.
        dificultad: El nivel de dificultad del habito ('Facil', 'Intermedio' o 'Dificil').
        descripcion: Detalles adicionales sobre el habito.
        tipo_habito: El tipo de habito que representa el objeto ('Positivo', 'Negativo' o 'Ambos').
        hitorial_positivo: Lista de fechas en la que un habito positivo fue completado.
        historial_negativo: Lista de fechas en la que un habito negativo fue completado.
        historial: Lista de fechas en las que un habito fue completada.
        contador_positivo: Numero de dias consecutivos realizando un habito positivo.
        contador_negativo: Numero de dias consecutivos realizando un habito negativo.
        fecha_creacion: Fecha en la que se creo un objeto de la clase Habito.
    '''
    # Se define una variable TIPOS_HABITO para cada tipo de habito
    TIPOS_HABITO = ["Positivo", "Negativo", "Ambos"]
    def __init__(self, titulo:str, dificultad:str, descripcion:str = None, tipo_habito = "Ambos"):
        # Verifica que el tipo de habito agregado sea uno válido
        if tipo_habito not in self.TIPOS_HABITO:
            raise ValueError(f"Se añadio un tipo de hábito no válido.\n Debe ser uno entre {self.TIPOS_HABITO}")
        super().__init__(titulo, dificultad, descripcion)
        self.tipo_habito = tipo_habito
        self.contador_positivo = 0
        self.historial_positivo = []
        self.contador_negativo = 0
        self.historial_negativo = []

    def calcular_recompensa(self, forma):
        # Verifica que se solicite la recompensa positiva o negativa de un Habito
        if forma not in ["Positivo", "Negativo"]:
            raise ValueError("La forma debe ser 'Positivo' o 'Negativo'.")
        recompensa = super().calcular_recompensa()
        # Modifica la recompensa original si el habito es Negaitvo
        if forma == "Negativo":
            recompensa["experiencia"] = recompensa["experiencia"]//2
            recompensa["oro"] = 0
            recompensa["vida"] = -1 * recompensa["vida"] - 2
        return recompensa

    @staticmethod
    def _calcular_racha(historial, contador):
        # Si no existe un historial de completaciones, la racha se vuelve 1
        if not historial:
            return 1
        ultima = datetime.strptime(historial[-1], FORMATO_FECHA).date()
        dias = (hoy()-ultima).days
        # Si la racha aumento hoy, se mantiene como estaba
        if dias == 0:
            return contador
        # Si la racha aumento ayer, se aumenta en 1 hoy
        if dias == 1:
            return contador + 1
        # Si la racha no se aumento ayer, se reinicia a 1
        return 1

    def marcar_positivo(self):
        # Verifica que el tipo de Habito sea 'Positivo' o 'Ambos'
        if self.tipo_habito in ["Positivo", "Ambos"]:
            self.contador_positivo = self._calcular_racha(self.historial_positivo, self.contador_positivo)
            hoy1 = str(hoy())
            self.historial_positivo.append(hoy1)
            self.historial.append(hoy1)
            # Devuelve una recompensa de Habito positiva
            return self.calcular_recompensa("Positivo")
        else:
            raise ValueError("El hábito no se completo. El Hábito no es del tipo <Hábito Positivo>.")
    
    def marcar_negativo(self):
        # Verifica que el tipo de Habito sea 'Negativo' o 'Ambos'
        if self.tipo_habito in ["Negativo", "Ambos"]:
            self.contador_negativo = self._calcular_racha(self.historial_negativo, self.contador_negativo)
            hoy1 = str(hoy())
            self.historial_negativo.append(hoy1)
            self.historial.append(hoy1)
            # Devuelve una recompensa de Habito negativa
            return self.calcular_recompensa("Negativo")
        else:
            raise ValueError("El hábito no se completo. El Hábito no es del tipo <Hábito Negativo>.")
        
    def completar(self):
        # Cada sentencia if verifica el tipo de Habito y devuelve una recompensa según corresponda
        if self.tipo_habito == "Positivo":
            return self.marcar_positivo()
        elif self.tipo_habito == "Negativo":
            return self.marcar_negativo()
        elif self.tipo_habito == "Ambos":
            return [self.marcar_positivo(), self.marcar_negativo()]

    def __repr__(self):
        # Devuelve información del Habito según el tipo.
        if self.tipo_habito in ["Positivo", "Negativo"]:
            recompensa = self.calcular_recompensa(self.tipo_habito)
            return f" ---------- Información sobre la Tarea ---------- \n| Título: {self.titulo} | Dificultad: {self.dificultad} | Descripcion: {self.descripcion} | Tipo de Habito: {self.tipo_habito} |Fecha de Creación: {self.fecha_creacion} | Nro. de Fechas Cumplidas: {len(self.historial)} | \n\n--- Información sobre la Recompensa ---\n| Experiencia: {recompensa["experiencia"]} | Oro: {recompensa["oro"]} | Vida: {recompensa["vida"]} |"
        else:
            recompensa_positiva = self.calcular_recompensa("Positivo")
            recompensa_negativa = self.calcular_recompensa("Negativo")
            return f" ---------- Información sobre la Tarea ---------- \n| Título: {self.titulo} | Dificultad: {self.dificultad} | Descripcion: {self.descripcion} | Tipo de Habito: {self.tipo_habito} |Fecha de Creación: {self.fecha_creacion} | Nro. de Fechas Cumplidas: {len(self.historial)} | \n\n--- Información sobre la Recompensa Positiva ---\n| Experiencia: {recompensa_positiva["experiencia"]} | Oro: {recompensa_positiva["oro"]} | Vida: {recompensa_positiva["vida"]} | \n\n--- Información sobre la Recompensa Negativa ---\n| Experiencia: {recompensa_negativa["experiencia"]} | Oro: {recompensa_negativa["oro"]} | Vida: {recompensa_negativa["vida"]} |"

class Diaria(Tarea):
    """Representa una tarea de un Jugador que se debe de cumplir una vez al día.
        
    Atributos:
        titulo: El nombre de la tarea.
        dificultad: El nivel de dificultad de la tarea ('Facil', 'Intermedio' o 'Dificil').
        descripcion: Detalles adicionales sobre la tarea.
        historial: Lista de fechas en las que la tarea fue completada.
        fecha_creacion: Fecha en la que se creo un objeto de la clase Tarea.
        racha: Número consecutivos de días en el que se completo la tarea Diaria.
        mejor_racha: Mayor número de días consecutivos en el que se completo la tarea Diaria.
        fecha_inicio_racha: La fecha en la que inicio la racha actual.
    """
    def __init__(self, titulo:str, dificultad:str, descripcion:str = None):
        super().__init__(titulo, dificultad, descripcion)
        self.racha = 0
        self.mejor_racha = 0
        self.fecha_inicio_racha = None

    def completar(self):
        # Aumenta la racha en uno
        self.racha += 1
        # Si no existe historial previo, establece la fecha de inicio de racha en hoy
        if len(self.historial) == 0:
            self.fecha_inicio_racha = str(hoy())
        # Si existe historial previo, verifica si la racha sigue vigente
        if len(self.historial) >= 2:
            if (datetime.strptime(self.historial[-1], FORMATO_FECHA) - datetime.strptime(self.historial[-2], FORMATO_FECHA)).days > 1:
                self.fecha_inicio_racha = str(hoy())
                self.racha = 1
        return super().completar()

    def fallar(self):
        # Verifica si la racha actual es mejor que la mejor racha
        if self.racha >= self.mejor_racha:
            self.mejor_racha = self.racha
        self.racha = 0
        recompensa = self.calcular_recompensa()
        recompensa["experiencia"] = 0
        recompensa["oro"] = 0
        recompensa["vida"] = -10
        return recompensa

class Pendiente(Tarea):
    """Representa una tarea pendiente con fecha limite <fecha_limite> de un Jugador.
    
    Atributos:
        titulo: El nombre de la tarea.
        dificultad: El nivel de dificultad de la tarea ('Facil', 'Intermedio' o 'Dificil').
        descripcion: Detalles adicionales sobre la tarea.
        historial: Lista de fechas en las que la tarea fue completada.
        fecha_creacion: Fecha en la que se creo un objeto de la clase Tarea.
        fecha_limite: Fecha máxima para completar la tarea. Pasada la fecha, la recompensa disminuira.
        completada: Estado de completado de la tarea pendiente.
        vencida: Estado de vencimiento de la tarea pendiente.
    """
    def __init__(self, titulo:str, dificultad:str, fecha_limite:str, descripcion:str = None):
        max = datetime.strptime(fecha_limite, FORMATO_FECHA).date()
        super().__init__(titulo, dificultad, descripcion)
        # Verifica si la fecha máxima para completar el Pendiente aún no ocurre
        if (max - hoy()).days >= 0:
            self.fecha_limite = fecha_limite
        else:
            raise ValueError("La fecha limite debe ser una igual o posterior a la de hoy.")
        self.completada = False
        self.vencida = False
        
    def esta_vencida(self):
        self.vencida = True

    def calcular_recompensa(self):
        recompensa = super().calcular_recompensa()
        # Verifica si el Pendiente esta vencido
        if self.vencida:
            recompensa["experiencia"] -= 4
            recompensa["oro"] -= 1
            recompensa["vida"] = 0
        return recompensa

# ---------------------------------------------------------------------------
# Calendario_Racha
# ---------------------------------------------------------------------------
class Calendario_Racha():
    """Representa el calendario de las rachas del Jugador en una Tarea.
        
    Atributos:
        tarea: Tarea de la cual se mostrara su calendario.
        inicio: Fecha de creacion de Tarea.
    """
    def __init__(self, tarea):
        # Verifica que el parametro clase tarea sea hijo de Tarea
        if not issubclass(type(tarea), Tarea):
            raise ValueError("El argumento <tarea> debe ser de la clase Tarea")
        self.tarea = tarea

    def mostrar(self, tipo_historial = None):
        # Verifica si se relleno el parametro tipo_historial para objetos que no son instancias de Habito.
        if not isinstance(self.tarea, Habito) and tipo_historial is not None:
            raise ValueError(f"El parametro <tipo_historial> solo debe ser ingresado para instancias de Habito.")
        # Verifica si no se relleno el parametro tipo_historial para objetos instancias de Habito.
        elif isinstance(self.tarea, Habito) and tipo_historial is None:
            raise ValueError(f"El parametro <tipo_historial> debe completarse para instancias de Habito.")
        tipos_historial = ["historial_positivo", "historial_negativo", "historial_completo"]
        # Verifica si el parametro tipo_historial no se completo
        if tipo_historial is None:
            tipo_historial = "historial_completo"
        # Verifica si el parametro tipo_historial es un tipo de historial válido
        if tipo_historial not in tipos_historial:
            raise ValueError(f"El tipo de historial ingresado no es un tipo válido para la clase Habito. El <tipo_historial> debe ser uno entre {tipos_historial}")
        
        if tipo_historial == "historial_completo":
            fechas_str = self.tarea.historial
        elif tipo_historial == "historial_positivo":
            fechas_str = self.tarea.historial_positivo
        elif tipo_historial == "historial_negativo":
            fechas_str = self.tarea.historial_negativo

        # Verifica si existe historial de completaciones
        if len(fechas_str) == 0:
            anio = int(self.inicio[:4])
            mes = int(self.inicio[5:7])
            print(calendar.month(anio, mes))
            return

        dict_fechas = {}
        for fecha in fechas_str:
            # Añade a dict_fechas cada fecha no repetida del historial de completaciones
            fecha_dt = datetime.strptime(fecha, FORMATO_FECHA)
            key_mes = (fecha_dt.year, fecha_dt.month)
            if key_mes not in dict_fechas.keys():
                dict_fechas[key_mes] = []
            dict_fechas[key_mes].append(str(fecha_dt.day))

        ults_tres_meses = sorted(dict_fechas.keys())[-3:]

        # Selecciona cada combinacion de año, mes
        for anio_mes in ults_tres_meses:
            anio, mes = anio_mes
            dias = dict_fechas[anio_mes]
            lineas = calendar.month(anio, mes).split("\n")

            # Colorea cada fecha en la que se completo la Tarea
            for i in range(2, len(lineas)):
                for dia in dias:
                    patron = rf"\b{dia}\b"
                    reemplazo = f"\033[91m{dia}\033[0m"
                    lineas[i] = re.sub(patron, reemplazo, lineas[i])
            print("\n".join(lineas))

In [6]:
# Ejecuta esta celda UNA sola vez (si la ejecutas de nuevo, se borran las cuentas y los clanes).
cuentas = []    # Cada cuenta guarda el jugador, sus hábitos, diarias, pendientes y su tienda
clanes = [ClanAbierto("Aventureros")]    # Los clanes los comparten todas las cuentas

In [7]:
# ---------------------------------------------------------------------------
# Menú de consola
# El usuario se mueve solo escribiendo el número de la opción.
# ---------------------------------------------------------------------------
from datetime import timedelta 
NIVEL_PARA_ELEGIR_CLASE = 5 # No modificar

# ----- Funciones para pedir datos -----
def pedir_numero(mensaje, minimo, maximo):
    while True: 
        texto = input(mensaje).strip()
        if texto.isdigit():
            numero = int(texto)
            if numero >= minimo and numero <= maximo:
                return numero
        print(f"Escribe un número entre {minimo} y {maximo}.")


def pedir_texto(mensaje):
    while True:
        texto = input(mensaje).strip()
        if texto != "":
            return texto
        print("No puedes dejarlo vacío.")

# Muestra las opciones numeradas
def mostrar_menu(titulo, opciones, texto_cero="Volver"):
    print()
    print(f"=== {titulo} ===")
    for i in range(len(opciones)):
        print(f"{i + 1}. {opciones[i]}")
    print(f"0. {texto_cero}")
    return pedir_numero("Escribe el número de la opción: ", 0, len(opciones))


# ----- Funciones de ayuda -----
def elegir_dificultad():
    """Devuelve 'Facil', 'Intermedio' o 'Dificil' (o None si el usuario cancela)."""
    numero = mostrar_menu("DIFICULTAD", ["Fácil", "Intermedio", "Difícil"], "Cancelar")
    if numero == 0:
        return None
    if numero == 1:
        return "Facil"
    if numero == 2:
        return "Intermedio"
    return "Dificil"


def elegir_tarea(titulo, tareas):
    """Muestra las tareas numeradas y devuelve la elegida (None si no hay o si cancela)."""
    if len(tareas) == 0:
        print("No tienes tareas de este tipo disponibles.")
        return None
    opciones = []
    for tarea in tareas:
        opciones.append(f"{tarea.titulo} ({tarea.dificultad})")
    numero = mostrar_menu(titulo, opciones, "Cancelar")
    if numero == 0:
        return None
    return tareas[numero - 1]


def mostrar_progreso(jugador, nivel_antes):
    """Muestra el estado del jugador después de una acción."""
    print(f"Nivel {jugador.nivel} | Experiencia {jugador.experiencia}/{jugador.exp_para_subir()} | "
          f"Vida {jugador.hp}/{jugador.hp_maxima} | Oro {jugador.oro}")
    if jugador.nivel > nivel_antes:
        print(f"¡Subiste al nivel {jugador.nivel}!")
    if nivel_antes < NIVEL_PARA_ELEGIR_CLASE and jugador.nivel >= NIVEL_PARA_ELEGIR_CLASE:
        print("¡Ya puedes elegir tu clase! Ve a 'Elegir mi clase' en el menú principal.")


# ----- Hábitos -----
def crear_habito(habitos):
    dificultad = elegir_dificultad()
    if dificultad is None:
        print("Cancelado.")
        return
    titulo = pedir_texto("Escribe el nombre del hábito: ")
    opciones = ["Positivo (algo bueno que quiero hacer)",
                "Negativo (algo malo que quiero evitar)",
                "Ambos"]
    numero = mostrar_menu("TIPO DE HÁBITO", opciones, "Cancelar")
    if numero == 0:
        print("Cancelado.")
        return
    if numero == 1:
        tipo = "Positivo"
    elif numero == 2:
        tipo = "Negativo"
    else:
        tipo = "Ambos"
    habitos.append(Habito(titulo, dificultad, None, tipo))
    print(f"Hábito creado: {titulo} ({dificultad}, {tipo}). Ahora tienes {len(habitos)} hábito(s).")


def listar_habitos(habitos):
    if len(habitos) == 0:
        print("Aún no tienes hábitos.")
        return
    print()
    for habito in habitos:
        print(f"- {habito.titulo} | {habito.dificultad} | Tipo: {habito.tipo_habito} | "
              f"Veces marcado: {len(habito.historial)}")


def marcar_habito(jugador, habitos, forma):
    """forma es 'Positivo' o 'Negativo'."""
    posibles = []
    for habito in habitos:
        if habito.tipo_habito == forma or habito.tipo_habito == "Ambos":
            posibles.append(habito)
    habito = elegir_tarea(f"MARCAR HÁBITO COMO {forma.upper()}", posibles)
    if habito is None:
        return
    nivel_antes = jugador.nivel
    jugador.marcar_habito(habito, forma)
    print(f"Marcaste '{habito.titulo}' como {forma.lower()}.")
    mostrar_progreso(jugador, nivel_antes)


def ver_calendario_habito(habitos):
    habito = elegir_tarea("CALENDARIO DE UN HÁBITO", habitos)
    if habito is None:
        return
    numero = mostrar_menu("¿QUÉ HISTORIAL QUIERES VER?", ["Positivo", "Negativo", "Completo"], "Cancelar")
    if numero == 0:
        return
    if numero == 1:
        historial = "historial_positivo"
    elif numero == 2:
        historial = "historial_negativo"
    else:
        historial = "historial_completo"
    print()
    Calendario_Racha(habito).mostrar(historial)


def menu_habitos(jugador, habitos):
    opciones = ["Crear un hábito",
                "Ver mis hábitos",
                "Marcar un hábito como positivo",
                "Marcar un hábito como negativo",
                "Ver el calendario de un hábito"]
    while True:
        opcion = mostrar_menu("HÁBITOS", opciones)
        if opcion == 0:
            return
        elif opcion == 1:
            crear_habito(habitos)
        elif opcion == 2:
            listar_habitos(habitos)
        elif opcion == 3:
            marcar_habito(jugador, habitos, "Positivo")
        elif opcion == 4:
            marcar_habito(jugador, habitos, "Negativo")
        else:
            ver_calendario_habito(habitos)


# ----- Diarias -----
def crear_diaria(diarias):
    dificultad = elegir_dificultad()
    if dificultad is None:
        print("Cancelado.")
        return
    titulo = pedir_texto("Escribe el nombre de la diaria: ")
    diarias.append(Diaria(titulo, dificultad))
    print(f"Diaria creada: {titulo} ({dificultad}). Ahora tienes {len(diarias)} diaria(s).")


def listar_diarias(diarias):
    if len(diarias) == 0:
        print("Aún no tienes diarias.")
        return
    print()
    for diaria in diarias:
        print(f"- {diaria.titulo} | {diaria.dificultad} | Racha: {diaria.racha} | "
              f"Mejor racha: {diaria.mejor_racha}")


def completar_diaria(jugador, diarias):
    diaria = elegir_tarea("COMPLETAR UNA DIARIA", diarias)
    if diaria is None:
        return
    nivel_antes = jugador.nivel
    if jugador.completar_tarea(diaria):
        print(f"¡Completaste '{diaria.titulo}'!")
        mostrar_progreso(jugador, nivel_antes)
    else:
        print("Ya completaste esta diaria hoy.")


def fallar_diaria(jugador, diarias):
    diaria = elegir_tarea("MARCAR UNA DIARIA COMO FALLADA", diarias)
    if diaria is None:
        return
    nivel_antes = jugador.nivel
    jugador.aplicar_recompensa(diaria.fallar())
    print(f"Fallaste '{diaria.titulo}'. Perdiste vida y tu racha volvió a 0.")
    mostrar_progreso(jugador, nivel_antes)


def ver_calendario_tarea(titulo, tareas):
    tarea = elegir_tarea(titulo, tareas)
    if tarea is None:
        return
    print()
    Calendario_Racha(tarea).mostrar()


def menu_diarias(jugador, diarias):
    opciones = ["Crear una diaria",
                "Ver mis diarias",
                "Completar una diaria",
                "Marcar una diaria como fallada",
                "Ver el calendario de una diaria"]
    while True:
        opcion = mostrar_menu("DIARIAS", opciones)
        if opcion == 0:
            return
        elif opcion == 1:
            crear_diaria(diarias)
        elif opcion == 2:
            listar_diarias(diarias)
        elif opcion == 3:
            completar_diaria(jugador, diarias)
        elif opcion == 4:
            fallar_diaria(jugador, diarias)
        else:
            ver_calendario_tarea("CALENDARIO DE UNA DIARIA", diarias)


# ----- Pendientes -----
def crear_pendiente(pendientes):
    dificultad = elegir_dificultad()
    if dificultad is None:
        print("Cancelado.")
        return
    titulo = pedir_texto("Escribe el nombre del pendiente: ")
    dias = pedir_numero("¿En cuántos días vence? (0 = hoy): ", 0, 365)
    fecha_limite = str(hoy() + timedelta(days=dias))
    pendientes.append(Pendiente(titulo, dificultad, fecha_limite))
    print(f"Pendiente creado: {titulo} ({dificultad}), vence el {fecha_limite}. "
          f"Ahora tienes {len(pendientes)} pendiente(s).")


def listar_pendientes(pendientes):
    if len(pendientes) == 0:
        print("Aún no tienes pendientes.")
        return
    print()
    for pendiente in pendientes:
        if pendiente.completada:
            estado = "Completado"
        else:
            estado = "Sin completar"
        print(f"- {pendiente.titulo} | {pendiente.dificultad} | Vence: {pendiente.fecha_limite} | {estado}")


def completar_pendiente(jugador, pendientes):
    sin_completar = []
    for pendiente in pendientes:
        if not pendiente.completada:
            sin_completar.append(pendiente)
    pendiente = elegir_tarea("COMPLETAR UN PENDIENTE", sin_completar)
    if pendiente is None:
        return
    nivel_antes = jugador.nivel
    jugador.completar_tarea(pendiente)
    print(f"¡Completaste '{pendiente.titulo}'!")
    mostrar_progreso(jugador, nivel_antes)


def menu_pendientes(jugador, pendientes):
    opciones = ["Crear un pendiente",
                "Ver mis pendientes",
                "Completar un pendiente",
                "Ver el calendario de un pendiente"]
    while True:
        opcion = mostrar_menu("PENDIENTES", opciones)
        if opcion == 0:
            return
        elif opcion == 1:
            crear_pendiente(pendientes)
        elif opcion == 2:
            listar_pendientes(pendientes)
        elif opcion == 3:
            completar_pendiente(jugador, pendientes)
        else:
            ver_calendario_tarea("CALENDARIO DE UN PENDIENTE", pendientes)


# ----- Tienda -----
def crear_tienda():
    tienda = Tienda()
    tienda.agregar_item(Pocion("Poción menor", 5, 1, "Restaura 15 de vida", 15))
    tienda.agregar_item(Pocion("Poción mayor", 12, 3, "Restaura 40 de vida", 40))
    tienda.agregar_item(Equipo("Armadura de cuero", 15, 2, "Aumenta la vida máxima", 10))
    tienda.agregar_item(Equipo("Armadura de acero", 40, 5, "Aumenta mucho la vida máxima", 25))
    tienda.agregar_item(Recompensa("Un capítulo de mi serie", 10, 1, "Premio personal", False))
    return tienda


def comprar_item(jugador, tienda):
    opciones = []
    for item in tienda.items:
        opciones.append(f"{item.nombre} - {item.precio} oro (nivel {item.nivel_requerido})")
    numero = mostrar_menu(f"COMPRAR (tienes {jugador.oro} de oro)", opciones, "Cancelar")
    if numero == 0:
        return
    item = tienda.items[numero - 1]
    tienda.comprar(jugador, item.codigo)


def crear_recompensa(tienda):
    nombre = pedir_texto("Escribe el nombre de tu recompensa: ")
    precio = pedir_numero("¿Cuánto oro cuesta? (0 a 1000): ", 0, 1000)
    tienda.agregar_item(Recompensa(nombre, precio, 1, "Recompensa personalizada", True))
    print("Recompensa agregada a la tienda.")


def menu_tienda(jugador, tienda):
    opciones = ["Ver los objetos de la tienda",
                "Comprar un objeto",
                "Crear una recompensa personal"]
    while True:
        opcion = mostrar_menu(f"TIENDA (tienes {jugador.oro} de oro)", opciones)
        if opcion == 0:
            return
        elif opcion == 1:
            tienda.mostrar_tabla()
        elif opcion == 2:
            comprar_item(jugador, tienda)
        else:
            crear_recompensa(tienda)


# ----- Clanes -----
def crear_clan(jugador, clanes):
    if jugador.clan is not None:
        print("Ya perteneces a un clan.")
        return
    nombre = pedir_texto("Escribe el nombre del clan: ")
    opciones = ["Abierto (cualquiera puede unirse)", "Privado (solo con invitación)"]
    numero = mostrar_menu("TIPO DE CLAN", opciones, "Cancelar")
    if numero == 0:
        print("Cancelado.")
        return
    if numero == 1:
        clan = ClanAbierto(nombre)
    else:
        clan = ClanPrivado(nombre)
        clan.invitar(jugador)
    clan.agregar_miembro(jugador)
    clanes.append(clan)
    print(f"Creaste el clan '{nombre}'.")


def unirse_a_clan(jugador, clanes):
    if jugador.clan is not None:
        print("Ya perteneces a un clan.")
        return
    abiertos = []
    for clan in clanes:
        if isinstance(clan, ClanAbierto):
            abiertos.append(clan)
    if len(abiertos) == 0:
        print("No hay clanes abiertos.")
        return
    opciones = []
    for clan in abiertos:
        opciones.append(f"{clan.nombre} ({len(clan.miembros)}/{clan.capacidad} miembros)")
    numero = mostrar_menu("CLANES ABIERTOS", opciones, "Cancelar")
    if numero == 0:
        return
    clan = abiertos[numero - 1]
    if clan.unirse(jugador):
        print(f"Te uniste al clan '{clan.nombre}'.")
    else:
        print("No pudiste unirte: el clan está lleno.")


def ver_mi_clan(jugador):
    if jugador.clan is None:
        print("No perteneces a ningún clan.")
    else:
        print()
        jugador.clan.mostrar_miembros()


def menu_clanes(jugador, clanes):
    opciones = ["Crear un clan", "Unirme a un clan abierto", "Ver mi clan"]
    while True:
        opcion = mostrar_menu("CLANES", opciones)
        if opcion == 0:
            return
        elif opcion == 1:
            crear_clan(jugador, clanes)
        elif opcion == 2:
            unirse_a_clan(jugador, clanes)
        else:
            ver_mi_clan(jugador)


# ----- Clase del personaje -----
# El jugador empieza como Aprendiz. Desde el nivel 5 puede elegir una clase 
def elegir_clase(jugador):
    if jugador.clase.nombre != "Aprendiz":
        print(f"Ya elegiste tu clase: {jugador.clase.nombre}.")
        return
    if jugador.nivel < NIVEL_PARA_ELEGIR_CLASE:
        print(f"Necesitas el nivel {NIVEL_PARA_ELEGIR_CLASE} para elegir tu clase. Tu nivel es {jugador.nivel}.")
        return
    clases = [Guerrero(), Mago(), Sanador(), Picaro()]
    opciones = []
    for clase in clases:
        opciones.append(str(clase))
    numero = mostrar_menu("ELIGE TU CLASE (no podrás cambiarla)", opciones, "Cancelar")
    if numero == 0:
        print("Sigues siendo Aprendiz.")
        return
    jugador.clase = clases[numero - 1]
    print(f"¡Ahora eres {jugador.clase.nombre}!")


# ----- Menú principal -----
def menu_principal(cuenta):
    # Estas listas son las mismas que guarda la cuenta: lo que se agrega aquí queda guardado en ella.
    # La lista 'clanes' viene de la celda de datos (la comparten todas las cuentas).
    jugador = cuenta["jugador"]
    habitos = cuenta["habitos"]
    diarias = cuenta["diarias"]
    pendientes = cuenta["pendientes"]
    tienda = cuenta["tienda"]
    opciones = ["Ver mi personaje",
                "Hábitos",
                "Diarias",
                "Pendientes",
                "Tienda",
                "Clanes",
                f"Elegir mi clase (desde el nivel {NIVEL_PARA_ELEGIR_CLASE})"]
    while True:
        opcion = mostrar_menu("MENÚ PRINCIPAL", opciones, "Cerrar sesión")
        if opcion == 0:
            print("Cerraste sesión. Tus datos quedaron guardados.")
            return
        elif opcion == 1:
            print()
            jugador.mostrar_datos()
        elif opcion == 2:
            menu_habitos(jugador, habitos)
        elif opcion == 3:
            menu_diarias(jugador, diarias)
        elif opcion == 4:
            menu_pendientes(jugador, pendientes)
        elif opcion == 5:
            menu_tienda(jugador, tienda)
        elif opcion == 6:
            menu_clanes(jugador, clanes)
        else:
            elegir_clase(jugador)


# ----- Cuentas -----
# Las cuentas se guardan en la lista 'cuentas' (celda anterior). Cada cuenta es un diccionario
# con el jugador y sus listas de hábitos, diarias, pendientes y su tienda.
def buscar_cuenta(nombre):
    """Busca una cuenta por nombre de usuario (sin importar mayúsculas). Devuelve None si no existe."""
    for cuenta in cuentas:
        if cuenta["jugador"].nombre.lower() == nombre.lower():
            return cuenta
    return None


def crear_cuenta(nivel_inicial=1):
    nombre = pedir_texto("Escribe tu nombre de usuario: ")
    if buscar_cuenta(nombre) is not None:
        print("Ese nombre de usuario ya existe. Prueba con otro o entra a tu cuenta.")
        return None
    jugador = Jugador(nombre, nivel_inicial)    # siempre empieza como Aprendiz
    cuenta = {"jugador": jugador,
              "habitos": [],
              "diarias": [],
              "pendientes": [],
              "tienda": crear_tienda()}
    cuentas.append(cuenta)
    print(f"¡Cuenta creada! Hola, {nombre}. Empiezas como {jugador.clase.nombre}.")
    return cuenta


def entrar_a_cuenta():
    if len(cuentas) == 0:
        print("Todavía no hay cuentas. Primero crea una.")
        return None
    nombre = pedir_texto("Escribe tu nombre de usuario: ")
    cuenta = buscar_cuenta(nombre)
    if cuenta is None:
        print("No existe una cuenta con ese nombre de usuario.")
        return None
    print(f"¡Bienvenido de nuevo, {cuenta['jugador'].nombre}!")
    return cuenta


def iniciar_juego(nivel_inicial=1):
    opciones = ["Entrar a mi cuenta", "Crear una cuenta nueva"]
    while True:
        opcion = mostrar_menu("BIENVENIDO A HABITICA", opciones, "Cerrar el juego")
        if opcion == 0:
            print("¡Hasta pronto!")
            return
        elif opcion == 1:
            cuenta = entrar_a_cuenta()
        else:
            cuenta = crear_cuenta(nivel_inicial)
        if cuenta is not None:
            menu_principal(cuenta)

In [8]:
iniciar_juego(8)


=== BIENVENIDO A HABITICA ===
1. Entrar a mi cuenta
2. Crear una cuenta nueva
0. Cerrar el juego
¡Cuenta creada! Hola, fiorella. Empiezas como Aprendiz.

=== MENÚ PRINCIPAL ===
1. Ver mi personaje
2. Hábitos
3. Diarias
4. Pendientes
5. Tienda
6. Clanes
7. Elegir mi clase (desde el nivel 5)
0. Cerrar sesión

Jugador: fiorella
Clase: Aprendiz
Nivel: 8
Experiencia: 0/400
Vida: 50/50
Oro: 0
Clan: Sin clan
Inventario: Vacío

=== MENÚ PRINCIPAL ===
1. Ver mi personaje
2. Hábitos
3. Diarias
4. Pendientes
5. Tienda
6. Clanes
7. Elegir mi clase (desde el nivel 5)
0. Cerrar sesión

=== ELIGE TU CLASE (no podrás cambiarla) ===
1. Guerrero: Muy resistente contra el daño
2. Mago: Sube de nivel más rápido
3. Sanador: Restaura su salud al subir de nivel
4. Picaro: Obtiene oro más rápido
0. Cancelar
¡Ahora eres Sanador!

=== MENÚ PRINCIPAL ===
1. Ver mi personaje
2. Hábitos
3. Diarias
4. Pendientes
5. Tienda
6. Clanes
7. Elegir mi clase (desde el nivel 5)
0. Cerrar sesión
Cerraste sesión. Tus datos q